In [ ]:
import pandas as pd
from pathlib import Path
from sqlalchemy import create_engine, inspect, text
df = pd.read_csv("../data/raw/customer_segment_data.csv")
df.head()
db_path = "../database/analytics.db"
engine = create_engine(f"sqlite:///{db_path}")

with engine.connect() as conn:
    print("Database connection successful")
    df.to_sql(
    "customers_cleaned",
    engine,
    if_exists="replace",
    index=False
)

print("Rows loaded:", len(df))

inspector = inspect(engine)

print("Tables:", inspector.get_table_names())

with engine.connect() as conn:
    count = conn.execute(
        text("SELECT COUNT(*) FROM customers_cleaned")
    ).scalar()

print("Row count:", count)

columns = inspector.get_columns("customers_cleaned")

for column in columns:
    print(column["name"], "->", column["type"])
    query = """
SELECT customer_id, customer_type, product, revenue, churn
FROM customers_cleaned
WHERE customer_type = 'Enterprise'
"""

enterprise_df = pd.read_sql(query, engine)
enterprise_df.head()
query = """
SELECT
    customer_type,
    COUNT(*) AS customer_count,
    AVG(revenue) AS average_revenue,
    SUM(revenue) AS total_revenue,
    AVG(churn) AS churn_rate
FROM customers_cleaned
GROUP BY customer_type
ORDER BY total_revenue DESC
"""

summary_df = pd.read_sql(query, engine)
summary_df
query = """
SELECT
    product,
    COUNT(*) AS customer_count,
    SUM(revenue) AS total_revenue,
    AVG(revenue) AS average_revenue
FROM customers_cleaned
GROUP BY product
ORDER BY total_revenue DESC
"""

product_df = pd.read_sql(query, engine)
product_df